# Stage 4. Pixel-level validation and exposure screeningRun once per target fire. The export half needs Earth Engine. The analysis halfreads the exported rasters and needs nothing else.

In [ ]:
import sysfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent / "src"))from fireanalogs.config import Configcfg = Config.load(Path.cwd().parent / "config" / "default.yaml")DATA = Path.cwd().parent / cfg["paths"]["data_root"]RESULTS = Path.cwd().parent / cfg["paths"]["results_root"]RESULTS.mkdir(parents=True, exist_ok=True)print("data:", DATA)

In [ ]:
FIRE_PREFIX = "20260111_265_streatham_yalla_y_poora_road_name_fi"RASTERS = DATA / "rasters"

## 4a. Export the target fire rastersThe predicted probability rasters are built by remapping the fuel by slope codeimage through the analog-derived cell probabilities, so the raster is exactlythe transfer function applied spatially. Run only when regenerating.

In [ ]:
# import ee, pandas as pd, numpy as np# ee.Initialize()## from fireanalogs import similarity# from fireanalogs.config import fuel_slope_columns# from fireanalogs.library import embedding_matrix# from fireanalogs.retrieval import candidate_pool, top_k_indices, cell_probabilities# from fireanalogs.gee.assets import datasets, fire_collection# from fireanalogs.gee.export import export_target_fire_rasters, wait_for_tasks## TARGET_FIRE_ID = "..."   # fire_id from the library## library = pd.read_csv(RESULTS / "fire_library_merged.csv", parse_dates=["event_date"])# cosine = similarity.cosine_similarity_matrix(embedding_matrix(library, cfg.n_emb_dims))# geo = similarity.haversine_km(library["lat"].values, library["lon"].values)## target = library.index[library["fire_id"] == TARGET_FIRE_ID][0]# pool = candidate_pool(target, library["event_date"].values, cfg.k_default, cfg.prior_only)# emb_analogs = top_k_indices(cosine[target], pool, cfg.k_default, higher_better=True)# geo_analogs = top_k_indices(-geo[target], pool, cfg.k_default, higher_better=True)## columns = fuel_slope_columns()# p_emb = cell_probabilities(emb_analogs, library, columns)# p_geo = cell_probabilities(geo_analogs, library, columns)## ds = datasets(cfg.raw)# fires = fire_collection(cfg.raw)# feature = ee.Feature(fires.filter(ee.Filter.eq("fire_id", TARGET_FIRE_ID)).first())# event_date = ee.Date(str(library.loc[target, "event_date"].date()))## tasks = export_target_fire_rasters(#     cfg.raw, ds, feature.geometry(), event_date, p_emb, p_geo,#     FIRE_PREFIX, "FIRE_EMBEDDINGS",# )# print(wait_for_tasks(tasks))

## 4b. Brier scores and spatial agreement

In [ ]:
from fireanalogs import pixelrasters = pixel.load_fire(RASTERS, FIRE_PREFIX, high_class_min=cfg.high_class_min)summary = pixel.brier_summary(rasters)summary

## Classification metrics and the threshold sweepThe published decision threshold is a reporting convention, not an optimum. Thesweep shows how the choice moves the trade-off between catching at-risk areaand raising false alarms.

In [ ]:
threshold = cfg["pixel_validation"]["decision_threshold"]classification = pixel.classification_metrics(rasters, threshold=threshold)sweep = pixel.threshold_sweep(rasters, cfg["pixel_validation"]["threshold_sweep"])sweep.to_csv(RESULTS / f"threshold_sweep_{FIRE_PREFIX}.csv", index=False)display(classification.round(3))sweep.round(3)

## Calibration across the fuel by slope grid

In [ ]:
calibration = pixel.calibration_by_fuel_slope(    rasters, min_pixels=cfg["pixel_validation"]["min_pixels_per_cell"])calibration.to_csv(RESULTS / "calibration_fuelslope.csv", index=False)calibration.round(3)

## Built-surface exposureExposure means coincidence of built surface with predicted high-severityprobability. It is not a damage, vulnerability or loss estimate.

In [ ]:
from fireanalogs import exposurebuilt = exposure.built_exposure(    rasters,    pixel_area_ha=cfg.pixel_area_ha,    threshold=cfg["exposure"]["built_threshold"],    fire_id=FIRE_PREFIX,)built.to_csv(RESULTS / f"built_exposure_{FIRE_PREFIX}.csv", index=False)outcome = exposure.outcome_analysis(    rasters,    pixel_area_ha=cfg.pixel_area_ha,    threshold=cfg["exposure"]["outcome_threshold"],    fire_id=FIRE_PREFIX,)outcome.to_csv(RESULTS / f"outcome_analysis_{FIRE_PREFIX}.csv", index=False)print("Note the two thresholds. Built-surface recall uses",      cfg["exposure"]["built_threshold"],      "and the whole-footprint outcome analysis uses",      cfg["exposure"]["outcome_threshold"])built.round(3)